# Graph-Based Anti-Money Laundering Detection
### IBM IT-AML Dataset (NeurIPS 2023) · LI-Small Split

**Objective:** Build a transaction-network-aware ML pipeline to detect money laundering accounts using graph features, XGBoost, and SHAP explainability.

**Dataset:** IBM Transactions for Anti-Money Laundering (AML) — a synthetic but research-grade dataset published at NeurIPS 2023, specifically designed to benchmark graph-based AML models. The LI (Lower Illicit ratio) split mimics realistic bank environments where laundering cases are rare.

**Approach:** Most AML systems flag individual transactions in isolation. The core insight of this project is that money laundering is a *network problem* — it's not about one transaction, it's about the flow across accounts. By building a transaction graph and computing structural features, we capture signals that row-level models miss.


## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set a clean, consistent plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

print("Libraries loaded.")

In [ ]:
# Load the two files that make up the LI-Small dataset
trans = pd.read_csv("/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv")
acc   = pd.read_csv("/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/LI-Small_accounts.csv")

print(f"Transactions shape : {trans.shape}")
print(f"Accounts shape     : {acc.shape}")
print()
print("Transaction columns:", trans.columns.tolist())
print()
print("Class balance:")
vc = trans['Is Laundering'].value_counts()
print(f"  Legitimate : {vc[0]:,}  ({vc[0]/len(trans)*100:.2f}%)")
print(f"  Laundering : {vc[1]:,}  ({vc[1]/len(trans)*100:.2f}%)")

**Initial observation:** The dataset is severely imbalanced — laundering cases make up less than 1% of all transactions. This is *realistic*. Real bank AML systems deal with exactly this problem. It means accuracy is a useless metric here (a model that predicts "legitimate" every time gets 99%+ accuracy but catches zero criminals). We'll use **AUC-ROC and F1-score** instead.

In [ ]:
trans.head()

In [ ]:
# Quick sanity check on the accounts file
acc.head()

## 2. Preprocessing & Time-Based Train/Test Split

One of the most common mistakes in ML projects is random train-test splitting on time-series data. Financial transactions have timestamps — if we randomly split, future transactions leak into training data. A model trained this way would look great on paper but completely fail in production.

**The right approach:** Sort by time, then split. Train on the past, validate on the present, test on the future. This is called a **temporal split** and it's exactly how banks validate their models before deployment.

In [ ]:
# Work on a clean copy
x = trans.copy()

# Parse timestamp
x['Timestamp'] = pd.to_datetime(x['Timestamp'], format='%Y/%m/%d %H:%M')

# Extract time components we'll use in EDA and features
x['hour']    = x['Timestamp'].dt.hour
x['minute']  = x['Timestamp'].dt.minute
x['day']     = x['Timestamp'].dt.day
x['weekday'] = x['Timestamp'].dt.weekday   # 0=Monday, 6=Sunday
x['Date']    = x['Timestamp'].dt.date

# Create unique account IDs by combining bank + account number
# This avoids collisions where account "12345" exists at two different banks
x['Sender_ID']   = x['From Bank'].astype(str) + '_' + x['Account'].astype(str)
x['Receiver_ID'] = x['To Bank'].astype(str)   + '_' + x['Account.1'].astype(str)

# Currency mismatch flag — a known AML red flag
x['is_conversion'] = (
    x['Payment Currency'].astype(str).str.strip() != 
    x['Receiving Currency'].astype(str).str.strip()
).astype(int)

# External hop flag — cross-bank transfers are riskier
x['is_external_hop'] = (x['From Bank'] != x['To Bank']).astype(int)

print("Preprocessing complete. New columns added:")
print(x[['Sender_ID', 'Receiver_ID', 'hour', 'weekday', 'is_conversion', 'is_external_hop']].head())

In [ ]:
# Sort chronologically — critical for temporal split
x = x.sort_values(by=['Date', 'hour', 'minute']).reset_index(drop=True)

# 60% train / 20% validation / 20% test — split by time, not randomly
n          = len(x)
train_end  = int(n * 0.60)
val_end    = int(n * 0.80)

train_df = x.iloc[:train_end].copy().reset_index(drop=True)
val_df   = x.iloc[train_end:val_end].copy().reset_index(drop=True)
test_df  = x.iloc[val_end:].copy().reset_index(drop=True)

print(f"Train : {len(train_df):,} rows  |  Laundering: {train_df['Is Laundering'].sum():,} ({train_df['Is Laundering'].mean()*100:.2f}%)")
print(f"Val   : {len(val_df):,} rows  |  Laundering: {val_df['Is Laundering'].sum():,} ({val_df['Is Laundering'].mean()*100:.2f}%)")
print(f"Test  : {len(test_df):,} rows  |  Laundering: {test_df['Is Laundering'].sum():,} ({test_df['Is Laundering'].mean()*100:.2f}%)")

# df = training set for EDA
df = train_df.copy()

## 3. Exploratory Data Analysis (EDA)

The goal of EDA here isn't just to make charts — it's to build *hypotheses* about how criminals behave differently from legitimate users. Each finding below directly maps to a feature we'll engineer later.

### 3.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = ['Legitimate', 'Laundering']
sizes  = [df['Is Laundering'].value_counts()[0], df['Is Laundering'].value_counts()[1]]
colors = ['#4C9BE8', '#E84C4C']
axes[0].pie(sizes, labels=labels, autopct='%1.2f%%', colors=colors,
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Transaction Class Distribution', fontsize=13, fontweight='bold')

# Bar with counts
axes[1].bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('Absolute Counts', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Transactions')
for i, v in enumerate(sizes):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Imbalance ratio: 1 laundering for every {round(sizes[0]/sizes[1])} legitimate transactions")

**Key takeaway:** The extreme imbalance (roughly 1 laundering case per 100–200 transactions in the train set) means we can't evaluate models by accuracy. A model predicting "legitimate" for everything would score ~99% accuracy but be completely useless. This is why we'll use **AUC-ROC, Precision-Recall curves, and weighted F1** as our metrics.

### 3.2 Temporal Patterns — When do criminals operate?

In [ ]:
fraud  = df[df['Is Laundering'] == 1]
normal = df[df['Is Laundering'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution comparison
sns.kdeplot(normal['hour'], label='Legitimate', fill=True, color='#4C9BE8',
            ax=axes[0], bw_adjust=1.5, alpha=0.6)
sns.kdeplot(fraud['hour'],  label='Laundering', fill=True, color='#E84C4C',
            ax=axes[0], bw_adjust=1.5, alpha=0.6)
axes[0].set_title('When Transactions Happen: Legitimate vs Laundering', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour of Day (0-23)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Fraud probability per hour
fraud_by_hour = df.groupby('hour')['Is Laundering'].mean()
axes[1].plot(fraud_by_hour.index, fraud_by_hour.values, 
             marker='o', color='darkred', linewidth=2, markersize=5)
axes[1].fill_between(fraud_by_hour.index, fraud_by_hour.values, color='red', alpha=0.1)
axes[1].axvspan(0, 4,   color='yellow', alpha=0.12, label='Elevated Risk Windows')
axes[1].axvspan(20, 23, color='yellow', alpha=0.12)
axes[1].set_title('P(Laundering | Hour) — Conditional Fraud Probability', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

**Insight:** Rather than operating at 3 AM when most fraud-detection rules look for anomalies, laundering transactions in this dataset cluster during business hours. This is a classic **"blend in" strategy** — criminals mimic legitimate payment behaviour to avoid rule-based filters that flag unusual timing. The early morning (0–4h) and late evening windows still show elevated fraud probability despite lower volume.

### 3.3 Transaction Amount — The Smurfing Signal

In [ ]:
eda_amt = df.copy()
bins   = [0, 1000, 2500, 5000, 7500, 10000, 25000, 50000, eda_amt['Amount Paid'].max()+1]
labels = ['0-1k', '1k-2.5k', '2.5k-5k', '5k-7.5k', '7.5k-10k', '10k-25k', '25k-50k', '50k+']

eda_amt['amount_bucket'] = pd.cut(eda_amt['Amount Paid'], bins=bins, labels=labels)
stats = eda_amt.groupby('amount_bucket', observed=False)['Is Laundering'].agg(['mean', 'count'])
stats.columns = ['fraud_rate', 'total_txns']

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(stats.index, stats['total_txns'], color='steelblue', alpha=0.3, label='Transaction Volume')
ax1.set_yscale('log')
ax1.set_ylabel('Transaction Volume (Log Scale)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xlabel('Transaction Amount ($)', fontsize=11)

ax2 = ax1.twinx()
ax2.plot(stats.index, stats['fraud_rate'], color='darkred', marker='o',
         linewidth=2.5, markersize=8, label='Fraud Probability')
ax2.set_ylabel('Fraud Probability', color='darkred', fontsize=11)
ax2.tick_params(axis='y', labelcolor='darkred')

# Highlight the smurfing zone
ax1.axvspan(-0.5, 1.5, color='orange', alpha=0.08, label='"Smurfing" Zone')

plt.title('Transaction Amount vs Fraud Risk — Identifying the Smurfing Zone', fontsize=13, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
fig.tight_layout()
plt.show()

**Insight — Smurfing detected:** There is a clear fraud spike in the **$1k–$5k range**. This statistically confirms *structuring behaviour* (commonly called "smurfing") — where criminals intentionally break large sums into smaller transactions to stay below the $10,000 reporting threshold that triggers automatic Currency Transaction Reports (CTRs). The second spike at $10k–$25k likely represents **layering-phase mule accounts**, where pre-washed funds are being moved in larger blocks using aged, trusted accounts.

### 3.4 Payment Format Risk

In [ ]:
format_stats = df.groupby('Payment Format', observed=False)['Is Laundering']                    .agg(['mean', 'count'])                    .rename(columns={'mean': 'fraud_rate', 'count': 'volume'})                    .sort_values('fraud_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(format_stats.index, format_stats['fraud_rate'],
               color=plt.cm.YlOrRd(format_stats['fraud_rate'] / format_stats['fraud_rate'].max()),
               edgecolor='grey', linewidth=0.5)
ax.set_xlabel('Fraud Probability P(Fraud | Payment Format)', fontsize=11)
ax.set_title('Which Payment Method do Criminals Prefer?', fontsize=13, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.4)

# Add volume annotations
for i, (_, row) in enumerate(format_stats.iterrows()):
    ax.text(row['fraud_rate'] + 0.0001, i, f"  n={row['volume']:,}", va='center', fontsize=9, color='grey')

plt.tight_layout()
plt.show()

**Insight:** ACH (Automated Clearing House) and Cheque formats show the highest fraud rates. This is counterintuitive — ACH is traditionally associated with "safe" recurring payments like payroll. The criminal strategy here is to exploit the low scrutiny attached to ACH by using it as a Trojan Horse. Wire transfers, despite being used for large sums, attract manual review and are therefore avoided by launderers.

### 3.5 Currency Conversion — Are criminals hopping currencies?

In [ ]:
eda_curr = df.copy()
eda_curr['Currency_Path'] = (eda_curr['Payment Currency'].str.strip() + 
                             " → " + eda_curr['Receiving Currency'].str.strip())

path_stats = eda_curr.groupby('Currency_Path')['Is Laundering']                      .agg(['mean', 'count'])                      .rename(columns={'mean': 'fraud_rate', 'count': 'volume'})
top_paths  = path_stats[path_stats['volume'] > 50].sort_values('fraud_rate', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart — top currency paths by risk
sns.barplot(x=top_paths['fraud_rate'], y=top_paths.index, palette='flare', ax=axes[0])
axes[0].set_title('Top Currency Paths by Fraud Risk', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fraud Probability')

# Conversion vs no-conversion comparison
conv_risk  = eda_curr.groupby('is_conversion')['Is Laundering'].mean()
bar_labels = ['Same Currency
(No Conversion)', 'Cross-Currency
(Conversion)']
bar_colors = ['#4C9BE8', '#E84C4C']
axes[1].bar(bar_labels, conv_risk.values, color=bar_colors, width=0.4, edgecolor='white')
axes[1].set_title('Same-Currency vs Cross-Currency Fraud Risk', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fraud Probability')
for i, v in enumerate(conv_risk.values):
    axes[1].text(i, v + 0.0002, f'{v:.4f}', ha='center', fontweight='bold')

multiplier = conv_risk.iloc[1] / conv_risk.iloc[0]
print(f"Cross-currency transactions are {multiplier:.2f}x riskier than same-currency transactions.")

plt.tight_layout()
plt.show()

**Insight:** Interestingly, *same-currency* flows dominate the laundering cases in this dataset. This makes sense — **intra-currency laundering is harder to detect**. Cross-currency conversions leave an obvious trail that compliance systems flag immediately. Sophisticated launderers avoid this and instead use structural (network-level) evasion rather than currency obfuscation. This finding reinforces why *graph topology* matters more than individual transaction metadata.

### 3.6 Bank-Level Risk — Identifying Mule Hubs

In [ ]:
bank_risk = df.groupby('To Bank')['Is Laundering']                .agg(['mean', 'count'])                .rename(columns={'mean': 'fraud_rate', 'count': 'volume'})
mule_hubs = bank_risk[bank_risk['volume'] > 100].sort_values('fraud_rate', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top receiving banks by fraud rate
sns.barplot(x=mule_hubs['fraud_rate'], y=mule_hubs.index.astype(str),
            palette='YlOrRd_r', ax=axes[0])
axes[0].set_title('Top 10 Destination Banks by Fraud Density', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fraud Rate')
axes[0].set_ylabel('Bank ID')

# Internal vs External hop risk
hop_risk   = df.groupby('is_external_hop')['Is Laundering'].mean()
hop_labels = ['Internal (Same Bank)', 'External (Cross-Bank)']
axes[1].bar(hop_labels, hop_risk.values, color=['#4C9BE8', '#E84C4C'], width=0.4, edgecolor='white')
axes[1].set_title('Internal vs Cross-Bank Transfer Risk', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fraud Probability')
multiplier_hop = hop_risk.iloc[1] / hop_risk.iloc[0]
for i, v in enumerate(hop_risk.values):
    axes[1].text(i, v + 0.0001, f'{v:.4f}', ha='center', fontweight='bold')
print(f"External transfers are {multiplier_hop:.2f}x riskier than internal ones.")

plt.tight_layout()
plt.show()

**Insight:** Certain bank IDs show fraud concentrations several times higher than average — these are **mule hubs**, institutions used as "landing zones" to aggregate illicit funds before the layering phase. Cross-bank (external) transfers are significantly riskier than internal ones, confirming that the laundering network deliberately moves money *across* institutions to obscure the trail.

## 4. Graph Construction

This is the core of the project and what makes it different from standard fraud detection notebooks.

**The idea:** Represent the entire financial system as a directed graph where:
- **Nodes** = unique bank accounts (identified by Bank + Account number)
- **Edges** = individual transactions (money flows from sender → receiver)

When we do this, money laundering patterns become *visible as graph structures*:
- **Fan-out:** One account sends to many different accounts → likely a "smurfing" source
- **Fan-in:** Many accounts send to one account → likely a collection mule
- **High betweenness:** An account sits in the middle of many shortest paths → likely a layering relay

None of these patterns are visible when looking at individual transaction rows.

In [ ]:
import networkx as nx

# Build the graph on the TRAINING data only — no leakage
print("Building transaction graph on training data...")
print(f"Processing {len(train_df):,} transactions...")

G = nx.DiGraph()

for _, row in train_df.iterrows():
    G.add_edge(
        row['Sender_ID'],
        row['Receiver_ID'],
        amount=row['Amount Received'],
        is_laundering=row['Is Laundering']
    )

print(f"\nGraph built:")
print(f"  Nodes (unique accounts) : {G.number_of_nodes():,}")
print(f"  Edges (transactions)    : {G.number_of_edges():,}")
print(f"  Graph density           : {nx.density(G):.6f}")
print(f"  Is directed             : {G.is_directed()}")

**Why a directed graph?** Money only flows one way. A transfer from Account A to Account B is fundamentally different from B to A. Using a directed graph preserves this asymmetry, which is critical for detecting layering chains (A → B → C → D) and distinguishing mule senders from mule receivers.

In [ ]:
# Extract node-level structural features
# These are the features that capture laundering patterns

print("Extracting graph features per account (this takes a few minutes)...")

in_deg   = dict(G.in_degree())
out_deg  = dict(G.out_degree())
in_wt    = dict(G.in_degree(weight='amount'))   # total money received
out_wt   = dict(G.out_degree(weight='amount'))  # total money sent

# Betweenness centrality — measures how often a node sits on the shortest
# path between two other nodes. High betweenness = relay/layering account.
# We use k=1000 approximation for speed (exact would take hours on full graph)
print("  Computing betweenness centrality (approximate, k=1000)...")
btw = nx.betweenness_centrality(G, k=min(1000, G.number_of_nodes()), normalized=True)

nodes = list(G.nodes())

node_features = pd.DataFrame({
    'account_id'        : nodes,
    'in_degree'         : [in_deg.get(n, 0)  for n in nodes],
    'out_degree'        : [out_deg.get(n, 0) for n in nodes],
    'total_received'    : [in_wt.get(n, 0)   for n in nodes],
    'total_sent'        : [out_wt.get(n, 0)  for n in nodes],
    'betweenness'       : [btw.get(n, 0)     for n in nodes],
})

# Derived features — engineered from the base graph metrics
node_features['degree_ratio']  = node_features['out_degree'] / (node_features['in_degree'] + 1)
node_features['amount_ratio']  = node_features['total_sent'] / (node_features['total_received'] + 1)
node_features['total_volume']  = node_features['total_received'] + node_features['total_sent']

# AML-specific flags
node_features['fan_out_flag']  = (node_features['out_degree'] > 5).astype(int)  # 1 → many (smurfing source)
node_features['fan_in_flag']   = (node_features['in_degree']  > 5).astype(int)  # many → 1 (collection mule)
node_features['relay_flag']    = ((node_features['in_degree'] > 1) & 
                                   (node_features['out_degree'] > 1)).astype(int)  # layering relay

print(f"\nFeature table shape: {node_features.shape}")
node_features.head()

In [ ]:
# Label each account: was it EVER involved in a laundering transaction?
laundering_senders   = set(train_df[train_df['Is Laundering'] == 1]['Sender_ID'])
laundering_receivers = set(train_df[train_df['Is Laundering'] == 1]['Receiver_ID'])
laundering_accounts  = laundering_senders | laundering_receivers

node_features['is_laundering_account'] = node_features['account_id'].isin(laundering_accounts).astype(int)

print("Account label distribution:")
vc2 = node_features['is_laundering_account'].value_counts()
print(f"  Legitimate accounts : {vc2[0]:,} ({vc2[0]/len(node_features)*100:.2f}%)")
print(f"  Laundering accounts : {vc2[1]:,} ({vc2[1]/len(node_features)*100:.2f}%)")

## 5. Feature Engineering

Before modelling, let's visualise what the graph features actually look like for laundering vs legitimate accounts. This step validates that our features carry a real signal.

In [ ]:
legit_nodes   = node_features[node_features['is_laundering_account'] == 0]
launder_nodes = node_features[node_features['is_laundering_account'] == 1]

features_to_plot = ['in_degree', 'out_degree', 'betweenness', 'degree_ratio']
titles = ['In-Degree
(Transactions Received)',
          'Out-Degree
(Transactions Sent)',
          'Betweenness Centrality
(Relay Score)',
          'Degree Ratio
(Out / In)']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, (feat, title) in enumerate(zip(features_to_plot, titles)):
    data_legit   = np.log1p(legit_nodes[feat])
    data_launder = np.log1p(launder_nodes[feat])
    
    axes[i].hist(data_legit,   bins=40, alpha=0.6, color='#4C9BE8', label='Legitimate', density=True)
    axes[i].hist(data_launder, bins=40, alpha=0.6, color='#E84C4C', label='Laundering', density=True)
    axes[i].set_title(title, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('log(1 + value)')
    axes[i].legend(fontsize=8)

plt.suptitle('Graph Feature Distributions: Laundering vs Legitimate Accounts', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Feature validation:** The distributions clearly separate in multiple dimensions — laundering accounts tend to have higher in-degree, higher out-degree, and elevated betweenness centrality. This confirms that structural graph features carry a real discriminatory signal that standard transaction-level features don't capture.

In [ ]:
# Correlation heatmap of graph features
feature_cols = ['in_degree', 'out_degree', 'total_received', 'total_sent',
                'betweenness', 'degree_ratio', 'amount_ratio',
                'fan_out_flag', 'fan_in_flag', 'relay_flag']

corr = node_features[feature_cols + ['is_laundering_account']].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Build test-set node features using the SAME graph 
# We extract features for val and test accounts from the already-built training graph
# (in production, you'd use a rolling graph — but for this project, this is the right approach)

def extract_node_features_from_graph(G, df_split, btw_scores):
    # Extract graph features for a set of accounts, using the training graph
    account_ids = list(set(df_split['Sender_ID'].tolist() + df_split['Receiver_ID'].tolist()))
    in_d  = dict(G.in_degree())
    out_d = dict(G.out_degree())
    in_w  = dict(G.in_degree(weight='amount'))
    out_w = dict(G.out_degree(weight='amount'))
    
    records = []
    for acc in account_ids:
        records.append({
            'account_id'     : acc,
            'in_degree'      : in_d.get(acc, 0),
            'out_degree'     : out_d.get(acc, 0),
            'total_received' : in_w.get(acc, 0),
            'total_sent'     : out_w.get(acc, 0),
            'betweenness'    : btw_scores.get(acc, 0),
        })
    
    feat_df = pd.DataFrame(records)
    feat_df['degree_ratio'] = feat_df['out_degree'] / (feat_df['in_degree'] + 1)
    feat_df['amount_ratio'] = feat_df['total_sent'] / (feat_df['total_received'] + 1)
    feat_df['total_volume'] = feat_df['total_received'] + feat_df['total_sent']
    feat_df['fan_out_flag'] = (feat_df['out_degree'] > 5).astype(int)
    feat_df['fan_in_flag']  = (feat_df['in_degree']  > 5).astype(int)
    feat_df['relay_flag']   = ((feat_df['in_degree'] > 1) & (feat_df['out_degree'] > 1)).astype(int)
    
    # Label
    launder_s = set(df_split[df_split['Is Laundering'] == 1]['Sender_ID'])
    launder_r = set(df_split[df_split['Is Laundering'] == 1]['Receiver_ID'])
    launder   = launder_s | launder_r
    feat_df['is_laundering_account'] = feat_df['account_id'].isin(launder).astype(int)
    
    return feat_df

val_features  = extract_node_features_from_graph(G, val_df,  btw)
test_features = extract_node_features_from_graph(G, test_df, btw)

print(f"Train features : {node_features.shape}")
print(f"Val features   : {val_features.shape}")
print(f"Test features  : {test_features.shape}")

## 6. Modelling

We train three models in order of complexity. The goal isn't just to get the best score — it's to show *why* the best model wins. Each model teaches us something:

1. **Logistic Regression** — the interpretable baseline. Fast, simple, tells us if the features are linearly separable.
2. **Random Forest** — an ensemble that handles non-linearities, gives us feature importance.
3. **XGBoost** — the gradient-boosting champion of tabular data. Most likely to win, but we justify it with evidence.

We use **SMOTE (Synthetic Minority Oversampling Technique)** to balance the training data. SMOTE creates synthetic examples of the minority class (laundering accounts) so the model doesn't just learn to predict "legitimate" for everything.

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.metrics         import (roc_auc_score, classification_report,
                                      RocCurveDisplay, precision_recall_curve,
                                      average_precision_score, confusion_matrix)
from sklearn.preprocessing   import StandardScaler
from imblearn.over_sampling  import SMOTE

FEATURES = ['in_degree', 'out_degree', 'total_received', 'total_sent',
            'betweenness', 'degree_ratio', 'amount_ratio', 'total_volume',
            'fan_out_flag', 'fan_in_flag', 'relay_flag']
TARGET   = 'is_laundering_account'

X_train = node_features[FEATURES].fillna(0)
y_train = node_features[TARGET]

X_val   = val_features[FEATURES].fillna(0)
y_val   = val_features[TARGET]

X_test  = test_features[FEATURES].fillna(0)
y_test  = test_features[TARGET]

# SMOTE on training set only
smote         = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Training set:")
print(f"  Legitimate : {(y_train_res == 0).sum():,}")
print(f"  Laundering : {(y_train_res == 1).sum():,}")
print()
print(f"Validation set (NO SMOTE — kept real distribution):")
print(f"  Legitimate : {(y_val == 0).sum():,}")
print(f"  Laundering : {(y_val == 1).sum():,}")

In [ ]:
# ── Model 1: Logistic Regression (Baseline) ──
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train_res)
X_val_sc    = scaler.transform(X_val)
X_test_sc   = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_res)
lr_val_auc = roc_auc_score(y_val, lr.predict_proba(X_val_sc)[:, 1])
print(f"Logistic Regression — Val AUC: {lr_val_auc:.4f}")

# ── Model 2: Random Forest ──
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
rf_val_auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
print(f"Random Forest      — Val AUC: {rf_val_auc:.4f}")

# ── Model 3: XGBoost (Champion) ──
xgb = XGBClassifier(
    n_estimators       = 300,
    max_depth          = 6,
    learning_rate      = 0.05,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = (y_train_res == 0).sum() / (y_train_res == 1).sum(),
    eval_metric        = 'auc',
    random_state       = 42,
    verbosity          = 0
)
xgb.fit(X_train_res, y_train_res,
        eval_set=[(X_val, y_val)], verbose=False)
xgb_val_auc = roc_auc_score(y_val, xgb.predict_proba(X_val)[:, 1])
print(f"XGBoost            — Val AUC: {xgb_val_auc:.4f}")

print(f"\nBest model on validation set: {'XGBoost' if xgb_val_auc >= max(lr_val_auc, rf_val_auc) else 'Random Forest'}")

### 6.1 Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models      = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb}
colors      = {'Logistic Regression': '#4C9BE8', 'Random Forest': '#2ECC71', 'XGBoost': '#E84C4C'}
val_inputs  = {'Logistic Regression': X_val_sc, 'Random Forest': X_val, 'XGBoost': X_val}

# ROC Curves
for name, model in models.items():
    X_in = val_inputs[name]
    RocCurveDisplay.from_estimator(model, X_in, y_val, name=name,
                                    ax=axes[0], color=colors[name])
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title('ROC Curve Comparison (Validation Set)', fontsize=12, fontweight='bold')

# Precision-Recall Curves
for name, model in models.items():
    X_in    = val_inputs[name]
    probs   = model.predict_proba(X_in)[:, 1]
    prec, rec, _ = precision_recall_curve(y_val, probs)
    ap      = average_precision_score(y_val, probs)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=colors[name], linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Validation Set)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Why Precision-Recall matters more than ROC-AUC here:** With extreme class imbalance, a model can achieve high AUC-ROC by being good at ranking legitimate transactions — even if it misses most laundering cases. The Precision-Recall curve is more honest: it shows the trade-off between catching criminals (recall) and not wasting investigator time on false alarms (precision). In an AML context, recall is usually prioritised — missing a criminal is worse than a false alarm.

### 6.2 Final Evaluation on Test Set

In [ ]:
# XGBoost on held-out test set (never seen during training or tuning)
test_probs = xgb.predict_proba(X_test)[:, 1]
test_preds = (test_probs > 0.5).astype(int)

test_auc   = roc_auc_score(y_test, test_probs)
test_ap    = average_precision_score(y_test, test_probs)

print("=" * 50)
print("XGBoost — FINAL TEST SET RESULTS")
print("=" * 50)
print(f"AUC-ROC            : {test_auc:.4f}")
print(f"Average Precision  : {test_ap:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, test_preds, target_names=['Legitimate', 'Laundering']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, test_preds)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted: Legit', 'Predicted: Launder'],
            yticklabels=['Actual: Legit', 'Actual: Launder'],
            linewidths=1, cbar=False)
ax.set_title('Confusion Matrix — XGBoost (Test Set)', fontsize=13, fontweight='bold')

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (legit correctly classified) : {tn:,}")
print(f"False Positives (legit wrongly flagged)      : {fp:,}")
print(f"False Negatives (launderers missed)          : {fn:,}")
print(f"True Positives  (launderers caught)          : {tp:,}")
print(f"\nFalse Positive Rate : {fp/(fp+tn)*100:.2f}% of legitimate accounts wrongly flagged")
print(f"Recall (Sensitivity): {tp/(tp+fn)*100:.2f}% of laundering accounts caught")

plt.tight_layout()
plt.show()

## 7. SHAP Explainability

In banking and finance, it's not enough to have a model that works — you need to be able to **explain every decision** to regulators and compliance teams. The EU's GDPR and financial regulators require models to provide reasons for decisions that affect customers.

SHAP (SHapley Additive exPlanations) gives each feature a contribution score for every single prediction. It answers the question: *"For this specific account, which features pushed the risk score up or down, and by how much?"*

In [ ]:
import shap

# Build SHAP explainer on the XGBoost model
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Global summary — what features matter most overall?
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False, plot_size=(10, 6))
plt.title('SHAP Summary Plot — Global Feature Importance', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

**Reading the SHAP summary plot:**
- Each dot is one account
- **Position on X-axis** = how much that feature pushed the prediction toward laundering (right) or away from it (left)
- **Color** = feature value (red = high, blue = low)

**Key findings:**
- `betweenness` at the top confirms that **accounts acting as relays** (high centrality) are the strongest laundering signal — this directly maps to the *layering* phase of money laundering
- `out_degree` being important confirms the **fan-out pattern** (smurfing sources)
- `in_degree` captures the **fan-in pattern** (collection mules)

This is exactly the kind of output a compliance officer at HSBC or Citi needs to justify flagging an account for investigation.

In [ ]:
# Bar chart version — cleaner for reporting
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES,
                  plot_type='bar', show=False, plot_size=(10, 6))
plt.title('Mean |SHAP| — Average Feature Impact on Predictions', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP for a single high-risk account — explainability at account level
test_probs_arr = xgb.predict_proba(X_test)[:, 1]
highest_risk_idx = int(np.argmax(test_probs_arr))

print(f"Highest risk account index: {highest_risk_idx}")
print(f"Predicted laundering probability: {test_probs_arr[highest_risk_idx]:.4f}")
print(f"Actual label: {'LAUNDERING' if y_test.iloc[highest_risk_idx] == 1 else 'Legitimate'}")
print()

# Waterfall plot for this specific account
shap.waterfall_plot(
    shap.Explanation(
        values       = shap_values[highest_risk_idx],
        base_values  = explainer.expected_value,
        data         = X_test.iloc[highest_risk_idx].values,
        feature_names= FEATURES
    )
)

**Interpreting the waterfall plot:** Starting from the base rate (average prediction), each bar shows how much a specific feature *pushed the risk score up or down* for this particular account. This is the output you'd show a compliance officer when explaining why an account was flagged — regulators require exactly this level of traceability.

## 8. Risk Tier Segmentation

Rather than giving a binary "flag / don't flag" output, real AML systems use **risk tiers**. This helps compliance teams prioritise their workload — Tier 1 gets immediate manual review, Tier 3 gets periodic monitoring. We segment accounts into three tiers based on their predicted laundering probability.

In [ ]:
# Get probabilities for all accounts we have features for
all_features  = pd.concat([node_features, val_features, test_features], ignore_index=True)
X_all         = all_features[FEATURES].fillna(0)
y_all         = all_features[TARGET]

all_probs = xgb.predict_proba(X_all)[:, 1]

# Risk tier segmentation
def assign_tier(p):
    if p >= 0.70:   return 'Tier 1 — High Risk (Immediate Review)'
    elif p >= 0.35: return 'Tier 2 — Medium Risk (Enhanced Monitoring)'
    else:           return 'Tier 3 — Low Risk (Standard Monitoring)'

all_features['risk_score'] = all_probs
all_features['risk_tier']  = all_features['risk_score'].apply(assign_tier)

tier_summary = all_features.groupby('risk_tier').agg(
    total_accounts=('account_id', 'count'),
    actual_laundering=('is_laundering_account', 'sum'),
    avg_risk_score=('risk_score', 'mean')
).round(4)
tier_summary['capture_rate'] = tier_summary['actual_laundering'] / all_features['is_laundering_account'].sum()

print("Risk Tier Summary:")
print(tier_summary.to_string())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

tier_colors = {
    'Tier 1 — High Risk (Immediate Review)' : '#E84C4C',
    'Tier 2 — Medium Risk (Enhanced Monitoring)': '#F39C12',
    'Tier 3 — Low Risk (Standard Monitoring)': '#4C9BE8'
}

tier_counts = all_features['risk_tier'].value_counts()
axes[0].pie(tier_counts.values, labels=[t.split('—')[1].strip() for t in tier_counts.index],
             colors=[tier_colors[t] for t in tier_counts.index],
             autopct='%1.1f%%', startangle=90, wedgeprops=dict(edgecolor='white'))
axes[0].set_title('Account Distribution by Risk Tier', fontsize=12, fontweight='bold')

# Risk score distribution
axes[1].hist(all_features[all_features['is_laundering_account'] == 0]['risk_score'],
             bins=60, alpha=0.6, color='#4C9BE8', label='Legitimate', density=True)
axes[1].hist(all_features[all_features['is_laundering_account'] == 1]['risk_score'],
             bins=60, alpha=0.7, color='#E84C4C', label='Laundering', density=True)
axes[1].axvline(0.35, color='orange', linestyle='--', linewidth=1.5, label='Tier 2 threshold (0.35)')
axes[1].axvline(0.70, color='red',    linestyle='--', linewidth=1.5, label='Tier 1 threshold (0.70)')
axes[1].set_xlabel('Predicted Laundering Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Risk Score Distribution', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Model Comparison Summary

In [ ]:
# Clean summary table
results = {
    'Model'             : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Val AUC-ROC'       : [round(lr_val_auc, 4), round(rf_val_auc, 4), round(xgb_val_auc, 4)],
    'Test AUC-ROC'      : [round(roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1]), 4),
                           round(roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]),    4),
                           round(test_auc, 4)],
    'Test Avg Precision': [round(average_precision_score(y_test, lr.predict_proba(X_test_sc)[:,1]), 4),
                           round(average_precision_score(y_test, rf.predict_proba(X_test)[:,1]),    4),
                           round(test_ap, 4)],
}
results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x   = np.arange(3)
w   = 0.25
ax.bar(x - w, results_df['Val AUC-ROC'],        width=w, label='Val AUC-ROC',        color='#4C9BE8')
ax.bar(x,     results_df['Test AUC-ROC'],       width=w, label='Test AUC-ROC',       color='#2ECC71')
ax.bar(x + w, results_df['Test Avg Precision'], width=w, label='Test Avg Precision', color='#E84C4C')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Conclusion & Key Findings

### What This Project Demonstrated

**1. Graph structure reveals what transaction rows hide.**
By modelling the financial network as a directed graph, we captured laundering patterns — fan-out, fan-in, and relay structures — that are completely invisible to row-level classifiers. The SHAP analysis confirmed that betweenness centrality (the relay signal) is the most important feature.

**2. Temporal data requires temporal splitting.**
Using a time-based 60/20/20 split instead of random splitting ensures no data leakage from the future into the training window. This is how real bank models are validated.

**3. Accuracy is the wrong metric for AML.**
With ~0.5% laundering prevalence, a "predict everything legitimate" model gets 99.5% accuracy but catches zero criminals. AUC-ROC, Precision-Recall, and recall-weighted F1 are the right measures.

**4. Explainability is not optional in finance.**
The SHAP waterfall plots show exactly why each account was flagged — this is a regulatory requirement (GDPR, SR 11-7) that most ML notebooks ignore.

### If I Were to Extend This Project
- **Temporal graph features** — rolling 7-day and 30-day graph stats per account to capture velocity changes
- **Node2Vec embeddings** — learn dense account representations from the graph topology
- **Model drift monitoring** — PSI (Population Stability Index) tracking to detect when the model starts degrading in production
- **Multi-hop path detection** — explicitly trace A→B→C→A circular chains using DFS graph traversal

### Limitations Acknowledged
- The IBM IT-AML dataset is synthetic — real bank data is messier, has more features, and often has incomplete labelling (many laundering transactions are never detected)
- Node-level SMOTE may introduce some bias since graph features of synthetic nodes aren't anchored to real network structures
- The betweenness centrality approximation (k=1000) introduces some estimation noise for very large graphs
